In [ ]:
import pandas as pd

df = pd.read_csv("customer-churn-training.csv")
df.head()

,customer_id,tenure_months,support_tickets,monthly_spend_inr,last_login_days,plan_type,churned
0,C001,2,4,499,21,Basic,1
1,C002,18,1,1299,2,Pro,0
2,C003,6,3,799,14,Basic,1
3,C004,30,0,1499,1,Pro,0
4,C005,11,2,999,5,Standard,0


In [ ]:
X = df.drop(["customer_id", "churned"], axis=1)
y = df["churned"]

print(X.head())

   tenure_months  support_tickets  monthly_spend_inr  last_login_days  \
0              2                4                499               21   
1             18                1               1299                2   
2              6                3                799               14   
3             30                0               1499                1   
4             11                2                999                5   

  plan_type  
0     Basic  
1       Pro  
2     Basic  
3       Pro  
4  Standard  


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(9, 5)
(3, 5)


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = [
    "tenure_months",
    "support_tickets",
    "monthly_spend_inr",
    "last_login_days"
]

categorical_features = [
    "plan_type"
]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Pipeline Created Successfully")

Pipeline Created Successfully


In [ ]:
from sklearn.ensemble import RandomForestClassifier

model_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])

model_pipeline.fit(X_train, y_train)

print("Training Complete")

Training Complete


In [ ]:
from sklearn.metrics import accuracy_score

predictions = model_pipeline.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)

Accuracy: 1.0


In [ ]:
corr = df.select_dtypes(include="number").corr()

print(corr)

                   tenure_months  support_tickets  monthly_spend_inr  \
tenure_months           1.000000        -0.893032           0.868524   
support_tickets        -0.893032         1.000000          -0.793155   
monthly_spend_inr       0.868524        -0.793155           1.000000   
last_login_days        -0.826201         0.941775          -0.817452   
churned                -0.760880         0.853786          -0.787426   

                   last_login_days   churned  
tenure_months            -0.826201 -0.760880  
support_tickets           0.941775  0.853786  
monthly_spend_inr        -0.817452 -0.787426  
last_login_days           1.000000  0.854060  
churned                   0.854060  1.000000  


In [ ]:
from sklearn.feature_selection import mutual_info_classif

X_encoded = pd.get_dummies(X, drop_first=True)

mi_scores = mutual_info_classif(
    X_encoded,
    y,
    random_state=42
)

importance = pd.Series(
    mi_scores,
    index=X_encoded.columns
).sort_values(ascending=False)

print(importance)

support_tickets       0.525433
monthly_spend_inr     0.404600
last_login_days       0.383766
tenure_months         0.364917
plan_type_Standard    0.124441
plan_type_Pro         0.087933
dtype: float64
